Limpeza e preparação dos dados brutos para análise.

In [6]:
import pandas as pd
import numpy as np
import warnings
from pathlib import Path

Diretórios

In [7]:
# Diretórios
DATA_DIR = Path("/content")

Carrega Dados

In [8]:

excel_file = DATA_DIR / "BASE DE DADOS PEDE 2024 - DATATHON.xlsx"

if not excel_file.exists():
    print(f"Arquivo não encontrado: {excel_file}")
    exit(1)

print(f"✓ Arquivo encontrado: {excel_file}")

# Listar abas
xls = pd.ExcelFile(excel_file)
print(f"\n✓ Abas disponíveis: {xls.sheet_names}")

# Carregar cada aba
dfs = {}
for sheet in xls.sheet_names:
    df = pd.read_excel(excel_file, sheet_name=sheet)
    dfs[sheet] = df
    print(f"  • {sheet}: {df.shape[0]} linhas × {df.shape[1]} colunas")

print()

✓ Arquivo encontrado: /content/BASE DE DADOS PEDE 2024 - DATATHON.xlsx

✓ Abas disponíveis: ['PEDE2022', 'PEDE2023', 'PEDE2024']
  • PEDE2022: 860 linhas × 42 colunas
  • PEDE2023: 1014 linhas × 48 colunas
  • PEDE2024: 1156 linhas × 50 colunas



Consolida Dados

In [10]:
# Adicionar coluna de ano
for sheet, df in dfs.items():
    if 'PEDE' in sheet:
        ano = int(sheet[-4:])  # Extrai ano do nome da aba (ex: PEDE2022 -> 2022)
        df['ANO'] = ano
        dfs[sheet] = df
        print(f"✓ {sheet}: Ano {ano} adicionado")

# Consolidar
df_consolidado = pd.concat(dfs.values(), ignore_index=True, sort=False)
print(f"\nDataset consolidado: {df_consolidado.shape[0]} linhas × {df_consolidado.shape[1]} colunas")
print()

✓ PEDE2022: Ano 2022 adicionado
✓ PEDE2023: Ano 2023 adicionado
✓ PEDE2024: Ano 2024 adicionado

Dataset consolidado: 3030 linhas × 64 colunas



Mapear e Padronizar Colunas

In [11]:
# Mapeamento de colunas para indicadores padrão
# Diferentes abas têm nomes ligeiramente diferentes
rename_dict = {
    'Idade 22': 'IDADE',
    'Idade': 'IDADE',
    'INDE 22': 'INDE_ANTERIOR',
    'INDE 23': 'INDE_ANTERIOR',
    'INDE 2023': 'INDE_ANTERIOR',
    'INDE 2024': 'INDE_ATUAL',
    'Matem': 'MATEMATICA',
    'Mat': 'MATEMATICA',
    'Portug': 'PORTUGUES',
    'Por': 'PORTUGUES',
    'Inglês': 'INGLES',
    'Ing': 'INGLES',
    'Fase ideal': 'FASE_IDEAL',
    'Fase Ideal': 'FASE_IDEAL',
    'Defas': 'DEFASAGEM',
    'Defasagem': 'DEFASAGEM',
}

df_consolidado = df_consolidado.rename(columns=rename_dict)

# Indicadores principais
indicadores = ['IAN', 'IDA', 'IEG', 'IAA', 'IPS', 'IPP', 'IPV']

print("Indicadores principais identificados:")
for ind in indicadores:
    if ind in df_consolidado.columns:
        print(f"  • {ind}: OK")
    else:
        print(f"  • {ind}: X NÃO ENCONTRADO")

# Adicionar INDE (usar INDE_ATUAL ou INDE_ANTERIOR)
if 'INDE_ATUAL' in df_consolidado.columns:
    df_consolidado['INDE'] = df_consolidado['INDE_ATUAL']
elif 'INDE_ANTERIOR' in df_consolidado.columns:
    df_consolidado['INDE'] = df_consolidado['INDE_ANTERIOR']

print("  • INDE: OK")

print()

✓ Indicadores principais identificados:
  • IAN: OK
  • IDA: OK
  • IEG: OK
  • IAA: OK
  • IPS: OK
  • IPP: OK
  • IPV: OK
  • INDE: OK



Converte Tipos de Dados Para Numerico

In [12]:
# Converter indicadores para numérico
for ind in indicadores + ['INDE']:
    if ind in df_consolidado.columns:
        missing_before = df_consolidado[ind].isnull().sum()

        # Converter para numérico
        df_consolidado[ind] = pd.to_numeric(df_consolidado[ind], errors='coerce')

        missing_after = df_consolidado[ind].isnull().sum()
        print(f"✓ {ind}: {missing_before} → {missing_after} valores faltantes")

print()

✓ IAN: 0 → 0 valores faltantes
✓ IDA: 178 → 178 valores faltantes
✓ IEG: 76 → 76 valores faltantes
✓ IAA: 165 → 165 valores faltantes
✓ IPS: 171 → 171 valores faltantes
✓ IPP: 1038 → 1038 valores faltantes
✓ IPV: 178 → 178 valores faltantes
✓ INDE: 1938 → 1976 valores faltantes



Tratamento de Missings com médias do ano - se não média geral

In [13]:
for ind in indicadores + ['INDE']:
    if ind in df_consolidado.columns:
        missing_before = df_consolidado[ind].isnull().sum()

        if missing_before > 0:
            # Preencher com média por ANO
            if 'ANO' in df_consolidado.columns:
                df_consolidado[ind] = df_consolidado.groupby('ANO')[ind].transform(
                    lambda x: x.fillna(x.mean())
                )

            # Se ainda houver NaN, preencher com média geral
            df_consolidado[ind].fillna(df_consolidado[ind].mean(), inplace=True)

            missing_after = df_consolidado[ind].isnull().sum()
            print(f"✓ {ind}: {missing_before} → {missing_after} valores faltantes")

print()

✓ IDA: 178 → 0 valores faltantes
✓ IEG: 76 → 0 valores faltantes
✓ IAA: 165 → 0 valores faltantes
✓ IPS: 171 → 0 valores faltantes
✓ IPP: 1038 → 0 valores faltantes
✓ IPV: 178 → 0 valores faltantes
✓ INDE: 1976 → 0 valores faltantes



/tmp/ipykernel_4795/1441675742.py:13: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_consolidado[ind].fillna(df_consolidado[ind].mean(), inplace=True)


Criação de features derivadas

In [21]:

# Feature 1: Média de indicadores
if all(ind in df_consolidado.columns for ind in indicadores):
    df_consolidado['INDICADORES_MEDIA'] = df_consolidado[indicadores].mean(axis=1)
    print("✓ INDICADORES_MEDIA: criado")

# Feature 2: Razão IEG/IDA
if 'IEG' in df_consolidado.columns and 'IDA' in df_consolidado.columns:
    df_consolidado['IEG_IDA_RATIO'] = df_consolidado['IEG'] / (df_consolidado['IDA'] + 0.001)
    print("✓ IEG_IDA_RATIO: criado")

# Feature 3: Razão INDE/IDA
if 'INDE' in df_consolidado.columns and 'IDA' in df_consolidado.columns:
    df_consolidado['INDE_IDA_RATIO'] = df_consolidado['INDE'] / (df_consolidado['IDA'] + 0.001)
    print("✓ INDE_IDA_RATIO: criado")

# Feature 4: Desalinhamento IAA-IDA
if 'IAA' in df_consolidado.columns and 'IDA' in df_consolidado.columns:
    df_consolidado['DESALINHAMENTO_IAA_IDA'] = df_consolidado['IAA'] - df_consolidado['IDA']
    print("✓ DESALINHAMENTO_IAA_IDA: criado")

# Feature 5: Categoria de risco (baseado em IAN)
if 'IAN' in df_consolidado.columns:
    df_consolidado['CATEGORIA_RISCO'] = pd.cut(
        df_consolidado['IAN'],
        bins=[0, 5, 10, 15, 20],
        labels=['Muito Alto Risco', 'Alto Risco', 'Risco Moderado', 'Baixo Risco'],
        include_lowest=True
    )
    print("✓ CATEGORIA_RISCO: criado")

# Feature 6: Fase normalizada
if 'Fase' in df_consolidado.columns:
    df_consolidado['FASE'] = df_consolidado['Fase']
    print("✓ FASE: normalizado")

print()
df_consolidado.head(5)

✓ INDICADORES_MEDIA: criado
✓ IEG_IDA_RATIO: criado
✓ INDE_IDA_RATIO: criado
✓ DESALINHAMENTO_IAA_IDA: criado
✓ CATEGORIA_RISCO: criado
✓ FASE: normalizado



,RA,Fase,Turma,Nome,Ano nasc,IDADE,Gênero,Ano ingresso,Instituição de ensino,Pedra 20,...,Escola,Ativo/ Inativo,Ativo/ Inativo.1,INDE,INDICADORES_MEDIA,IEG_IDA_RATIO,INDE_IDA_RATIO,DESALINHAMENTO_IAA_IDA,CATEGORIA_RISCO,FASE
0,RA-1,7,A,Aluno-1,2003.0,19.0,Menina,2016,Escola Pública,Ametista,...,NaN,NaN,NaN,7.396686,5.976164,1.024744,1.848709,4.3,Muito Alto Risco,7
1,RA-2,7,A,Aluno-2,2005.0,17.0,Menina,2017,Rede Decisão,Ametista,...,NaN,NaN,NaN,7.396686,7.347593,0.764593,1.087588,2.0,Alto Risco,7
2,RA-3,7,A,Aluno-3,2005.0,17.0,Menina,2016,Rede Decisão,Ametista,...,NaN,NaN,NaN,7.396686,6.315879,1.410462,1.320601,-5.6,Alto Risco,7
3,RA-4,7,A,Aluno-4,2005.0,17.0,Menino,2017,Rede Decisão,Ametista,...,NaN,NaN,NaN,7.396686,6.676164,0.899820,1.479041,3.8,Alto Risco,7
4,RA-5,7,A,Aluno-5,2005.0,17.0,Menina,2016,Rede Decisão,Ametista,...,NaN,NaN,NaN,7.396686,7.463450,1.653528,1.422166,2.7,Alto Risco,7


Descritivas e estatisticas

In [20]:
print("\nResumo dos indicadores principais:")
print(df_consolidado[indicadores + ['INDE']].describe().round(2))

print("\nDistribuição por ano:")
if 'ANO' in df_consolidado.columns:
    print(df_consolidado['ANO'].value_counts().sort_index())

print()

📊 ETAPA 7: Estatísticas descritivas...
--------------------------------------------------------------------------------

Resumo dos indicadores principais:
           IAN      IDA      IEG      IAA      IPS      IPP      IPV     INDE
count  3030.00  3030.00  3030.00  3030.00  3030.00  3030.00  3030.00  3030.00
mean      7.18     6.38     7.96     7.92     6.28     7.56     7.55     7.40
std       2.54     1.90     2.13     2.56     1.75     0.76     1.06     0.60
min       2.50     0.00     0.00     0.00     2.50     2.50     2.50     3.79
25%       5.00     5.20     7.40     7.90     5.12     7.50     7.01     7.40
50%       5.00     6.60     8.61     8.54     6.90     7.56     7.56     7.40
75%      10.00     7.80     9.40     9.50     7.51     7.81     8.22     7.40
max      10.00    10.00    10.00    10.00    10.00    10.00    10.01     9.53

Distribuição por ano:
ANO
2022     860
2023    1014
2024    1156
Name: count, dtype: int64



Salva Dados Tratados

In [22]:
# Selecionar colunas principais para salvar
colunas_principais = ['RA', 'ANO', 'Fase', 'IDADE', 'Gênero', 'Instituição de ensino'] + indicadores + ['INDE'] + [
    'INDICADORES_MEDIA', 'IEG_IDA_RATIO', 'INDE_IDA_RATIO', 'DESALINHAMENTO_IAA_IDA', 'CATEGORIA_RISCO'
]

colunas_para_salvar = [col for col in colunas_principais if col in df_consolidado.columns]
df_para_salvar = df_consolidado[colunas_para_salvar].copy()

# Salvar dataset completo
output_path = DATA_DIR / "df_clean.csv"
df_para_salvar.to_csv(output_path, index=False, encoding='utf-8')
print(f"✓ Dataset completo salvo: {output_path}")

# Salvar por ano
if 'ANO' in df_para_salvar.columns:
    for ano in sorted(df_para_salvar['ANO'].unique()):
        if pd.notna(ano):
            df_ano = df_para_salvar[df_para_salvar['ANO'] == ano]
            ano_path = DATA_DIR / f"df_{int(ano)}_clean.csv"
            df_ano.to_csv(ano_path, index=False, encoding='utf-8')
            print(f"✓ Dataset {int(ano)} salvo: {ano_path} ({len(df_ano)} registros)")

print()

✓ Dataset completo salvo: /content/df_clean.csv
✓ Dataset 2022 salvo: /content/df_2022_clean.csv (860 registros)
✓ Dataset 2023 salvo: /content/df_2023_clean.csv (1014 registros)
✓ Dataset 2024 salvo: /content/df_2024_clean.csv (1156 registros)



Relátorio Final dos Dados

In [24]:

print(f"  • Total de registros: {len(df_para_salvar):,}")
print(f"  • Total de colunas: {len(df_para_salvar.columns)}")
print(f"  • Indicadores processados: {len(indicadores) + 1}")
print(f"  • Features derivadas: 5")
print(f"  • Anos cobertos: {sorted(df_para_salvar['ANO'].unique())}")
print(f"  • Valores faltantes restantes: {df_para_salvar[indicadores + ['INDE']].isnull().sum().sum()}")


  • Total de registros: 3,030
  • Total de colunas: 20
  • Indicadores processados: 8
  • Features derivadas: 5
  • Anos cobertos: [np.int64(2022), np.int64(2023), np.int64(2024)]
  • Valores faltantes restantes: 0
